# Student Health Risk

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

## Load the dataset

In [ ]:
df = pd.read_csv('dataset/train.csv')

## Data Understanding

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
summary_rows = []

n_rows = len(df)

for column_name, column_data in df.items():

    missing_count = column_data.isna().sum()
    missing_ratio = (missing_count / n_rows) * 100

    example_values = column_data.dropna().unique()[:3]
    example_preview = ", ".join(map(str, example_values))[:60]

    summary_rows.append({
        "column_name": column_name,
        "dtype": column_data.dtype,
        "unique_values": column_data.nunique(),
        "missing_count": missing_count,
        "missing_percent": round(missing_ratio, 2),
        "example_values": example_preview
    })

dataset_summary = pd.DataFrame(summary_rows)

display(
    dataset_summary.style.set_properties(**{"text-align": "left"})
)

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nMissing values:")
print(df.isnull().sum())

print("\nMissing-value percentage:")
print(
    (df.isnull().mean() * 100)
    .sort_values(ascending=False)
    .round(2)
)

In [ ]:
df.describe(include='all').T

In [ ]:
numerical_columns = df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = df.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

In [ ]:
for column in categorical_columns:
    print(f"\nColumn: {column}")
    print(df[column].value_counts(dropna=False))

In [ ]:
for column in categorical_columns:
    df[column] = (
        df[column]
        .str.strip()
        .str.lower()
    )

In [ ]:
numerical_columns.remove("id")

In [ ]:
categorical_columns.remove("health_condition")

In [ ]:
df.duplicated().sum()

In [ ]:
df.duplicated(
    subset=df.columns.drop("id")
).sum()

In [ ]:
df.drop(columns=["id"], inplace=True)

## EDA: Exploratory Data Analysis

### Target Variable Analysis

In [ ]:
target_percentage = (
    df["health_condition"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

target_percentage

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="health_condition"
)

plt.title("Distribution of Student Health Conditions")
plt.xlabel("Health Condition")
plt.ylabel("Number of Students")

plt.show()

### Numerical Features Analysis

In [ ]:
df[numerical_columns].hist(
    figsize=(16, 12),
    bins=30
)

plt.suptitle(
    "Distribution of Numerical Health Features",
    fontsize=16
)

plt.tight_layout()
plt.show()

### Outlier Analysis

In [ ]:
for column in numerical_columns:

    plt.figure(figsize=(8, 4))

    sns.boxplot(
        data=df,
        x=column
    )

    plt.title(
        f"Outlier Analysis: {column}"
    )

    plt.show()

In [ ]:
df[numerical_columns].agg(["min", "max", "mean", "median"]).T

### Missing Value Analysis

In [ ]:
missing_data = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

missing_data = (
    missing_data[
        missing_data["Missing_Count"] > 0
    ]
    .sort_values(
        "Missing_Percentage",
        ascending=False
    )
)

missing_data

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=missing_data.reset_index(),
    x="Missing_Percentage",
    y="index"
)

plt.title("Percentage of Missing Values")
plt.xlabel("Missing Values (%)")
plt.ylabel("Feature")

plt.show()

### Categorical Features Analysis

In [ ]:
for column in categorical_columns:

    plt.figure(figsize=(8, 5))

    order = (
        df[column]
        .value_counts()
        .index
    )

    sns.countplot(
        data=df,
        x=column,
        order=order
    )

    plt.title(
        f"Distribution of {column.replace('_', ' ').title()}"
    )

    plt.xlabel(
        column.replace("_", " ").title()
    )

    plt.ylabel("Number of Students")

    plt.xticks(
        rotation=30
    )

    plt.tight_layout()
    plt.show()

### Analyze categorical features against health condition

In [ ]:
for column in categorical_columns:

    percentage_table = pd.crosstab(
        df[column],
        df["health_condition"],
        normalize="index"
    ) * 100

    percentage_table.plot(
        kind="bar",
        stacked=True,
        figsize=(9, 5)
    )

    plt.title(
        f"Health Condition Percentage by "
        f"{column.replace('_', ' ').title()}"
    )

    plt.xlabel(
        column.replace("_", " ").title()
    )

    plt.ylabel(
        "Percentage of Students"
    )

    plt.xticks(
        rotation=30
    )

    plt.legend(
        title="Health Condition"
    )

    plt.tight_layout()
    plt.show()

### Numerical features versus health condition

In [ ]:
for column in numerical_columns:

    plt.figure(figsize=(9, 5))

    sns.boxplot(
        data=df,
        x="health_condition",
        y=column
    )

    plt.title(
        f"{column.replace('_', ' ').title()} "
        "by Health Condition"
    )

    plt.xlabel(
        "Health Condition"
    )

    plt.ylabel(
        column.replace("_", " ").title()
    )

    plt.show()

### Comparing group statistics

In [ ]:
health_summary = (
    df.groupby(
        "health_condition"
    )[numerical_columns]
    .agg([
        "mean",
        "median"
    ])
    .round(2)
)

health_summary

In [ ]:
mean_comparison = (
    df.groupby(
        "health_condition"
    )[numerical_columns]
    .mean()
    .round(2)
)

mean_comparison.T

### Correlation analysis

In [ ]:
correlation_matrix = (
    df[numerical_columns]
    .corr()
)

plt.figure(
    figsize=(10, 8)
)

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title(
    "Correlation Between Numerical Health Features"
)

plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix

## Feature engineering

### Creating features

In [ ]:
def create_features(data):

    data = data.copy()

    # BMI category
    data["bmi_category"] = pd.cut(
        data["bmi"],
        bins=[0, 18.5, 25, 30, np.inf],
        labels=[
            "underweight",
            "normal",
            "overweight",
            "obese"
        ]
    )

    # Sleep category
    data["sleep_category"] = pd.cut(
        data["sleep_duration"],
        bins=[0, 6, 8, np.inf],
        labels=[
            "insufficient",
            "recommended",
            "long"
        ]
    )

    # Daily step category
    data["step_category"] = pd.cut(
        data["step_count"],
        bins=[0, 5000, 10000, np.inf],
        labels=[
            "low",
            "moderate",
            "high"
        ]
    )

    # Exercise category
    data["exercise_category"] = pd.cut(
        data["exercise_duration"],
        bins=[-1, 20, 45, np.inf],
        labels=[
            "low",
            "moderate",
            "high"
        ]
    )

    # Hydration category
    data["hydration_category"] = pd.cut(
        data["water_intake"],
        bins=[0, 1.5, 2.5, np.inf],
        labels=[
            "low",
            "adequate",
            "high"
        ]
    )

    # Combined activity score
    data["activity_score"] = (
        data["step_count"] *
        data["exercise_duration"]
    )

    # Interaction between sleep and exercise
    data["sleep_activity_interaction"] = (
        data["sleep_duration"] *
        data["exercise_duration"]
    )

    return data

In [ ]:
df_featured = create_features(df)

In [ ]:
df_featured.head()

In [ ]:
df_featured.shape

### Separating features and target

In [ ]:
X = df_featured.drop(
    columns="health_condition"
)

y = df_featured[
    "health_condition"
]

In [ ]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## Train-test split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print(
    "Training shape:",
    X_train.shape
)

print(
    "Testing shape:",
    X_test.shape
)

In [ ]:
print(
    "Original distribution:"
)

print(
    y.value_counts(
        normalize=True
    ).round(3)
)

print(
    "\nTraining distribution:"
)

print(
    y_train.value_counts(
        normalize=True
    ).round(3)
)

print(
    "\nTesting distribution:"
)

print(
    y_test.value_counts(
        normalize=True
    ).round(3)
)

In [ ]:
numerical_features = (
    X_train
    .select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(
        exclude=["number"]
    )
    .columns
    .tolist()
)

In [ ]:
print(
    "Numerical features:"
)

print(
    numerical_features
)

print(
    "\nCategorical features:"
)

print(
    categorical_features
)

In [ ]:
from sklearn.pipeline import Pipeline

from sklearn.compose import (
    ColumnTransformer
)

from sklearn.impute import (
    SimpleImputer
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

In [ ]:
# numerical preprocessing pipeline
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median" # fill missing values with median for numerical features
            )
        ),

        (
            "scaler",
            StandardScaler() # scale numerical features
        )
    ]
)

# categorical preprocessing pipeline
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent" # fill missing values with the most frequent value for categorical features
            )
        ),

        (
            "encoder", # encode categorical features using one-hot encoding
            OneHotEncoder(
                handle_unknown="ignore" # ignore unknown categories during encoding
            )
        )
    ]
)

# Combine numerical and categorical pipelines into a single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

## Building a baseline model

In [ ]:
from sklearn.linear_model import (
    LogisticRegression
)

In [ ]:
logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

In [ ]:
logistic_model.fit(
    X_train,
    y_train
)

In [ ]:
y_pred_logistic = (
    logistic_model.predict(
        X_test
    )
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

In [ ]:
print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test,
            y_pred_logistic
        ),
        4
    )
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            y_pred_logistic
        ),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_test,
            y_pred_logistic,
            average="macro"
        ),
        4
    )
)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_logistic
    )
)

In [ ]:
labels = [
    "fit",
    "at-risk",
    "unhealthy"
]

cm = confusion_matrix(
    y_test,
    y_pred_logistic,
    labels=labels
)

plt.figure(
    figsize=(8, 6)
)

sns.heatmap(
    cm,
    annot=True,
    fmt=",",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.title(
    "Logistic Regression Confusion Matrix"
)

plt.xlabel(
    "Predicted Health Condition"
)

plt.ylabel(
    "Actual Health Condition"
)

plt.show()

## Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=20,
                min_samples_split=10,
                min_samples_leaf=5,
                class_weight="balanced",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

In [ ]:
random_forest_model.fit(
    X_train,
    y_train
)

In [ ]:
y_pred_rf = random_forest_model.predict(
    X_test
)

In [ ]:
print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test,
            y_pred_rf
        ),
        4
    )
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            y_pred_rf
        ),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_test,
            y_pred_rf,
            average="macro"
        ),
        4
    )
)

print(
    "\nClassification Report:\n"
)

print(
    classification_report(
        y_test,
        y_pred_rf
    )
)

In [ ]:
labels = [
    "fit",
    "at-risk",
    "unhealthy"
]

rf_cm = confusion_matrix(
    y_test,
    y_pred_rf,
    labels=labels
)

plt.figure(
    figsize=(8, 6)
)

sns.heatmap(
    rf_cm,
    annot=True,
    fmt=",",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.title(
    "Random Forest Confusion Matrix"
)

plt.xlabel(
    "Predicted Health Condition"
)

plt.ylabel(
    "Actual Health Condition"
)

plt.show()

## Compare Both Models

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "Accuracy": [
        accuracy_score(
            y_test,
            y_pred_logistic
        ),

        accuracy_score(
            y_test,
            y_pred_rf
        )
    ],

    "Balanced Accuracy": [
        balanced_accuracy_score(
            y_test,
            y_pred_logistic
        ),

        balanced_accuracy_score(
            y_test,
            y_pred_rf
        )
    ],

    "Macro F1": [
        f1_score(
            y_test,
            y_pred_logistic,
            average="macro"
        ),

        f1_score(
            y_test,
            y_pred_rf,
            average="macro"
        )
    ]
})

model_comparison.round(4)

### Top 15 Predictors of Student Health Condition

In [ ]:
feature_names = (
    random_forest_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

feature_importance = pd.DataFrame({
    "Feature": feature_names,

    "Importance":
    random_forest_model
    .named_steps["classifier"]
    .feature_importances_
})

In [ ]:
feature_importance["Feature"] = (
    feature_importance["Feature"]
    .str.replace(
        "numerical__",
        "",
        regex=False
    )
    .str.replace(
        "categorical__",
        "",
        regex=False
    )
)

In [ ]:
top_features = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .head(15)
)

top_features

In [ ]:
plt.figure(
    figsize=(10, 7)
)

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title(
    "Top 15 Predictors of Student Health Condition"
)

plt.xlabel(
    "Random Forest Feature Importance"
)

plt.ylabel(
    "Feature"
)

plt.tight_layout()
plt.show()

### Checking overfitting

In [ ]:
y_train_pred_rf = random_forest_model.predict(
    X_train
)

print(
    "Training Accuracy:",
    round(
        accuracy_score(
            y_train,
            y_train_pred_rf
        ),
        4
    )
)

print(
    "Testing Accuracy:",
    round(
        accuracy_score(
            y_test,
            y_pred_rf
        ),
        4
    )
)

print(
    "Training Macro F1:",
    round(
        f1_score(
            y_train,
            y_train_pred_rf,
            average="macro"
        ),
        4
    )
)

print(
    "Testing Macro F1:",
    round(
        f1_score(
            y_test,
            y_pred_rf,
            average="macro"
        ),
        4
    )
)

## Permutation Importance

Permutation importance is often more reliable than built-in Random Forest importance because it measures how much model performance decreases when a feature is shuffled.

In [ ]:
from sklearn.inspection import (
    permutation_importance
)

Using a small sample of data to reduce computation time

In [ ]:
X_test_sample = X_test.sample(
    n=20000,
    random_state=42
)

y_test_sample = y_test.loc[
    X_test_sample.index
]

In [ ]:
permutation_result = permutation_importance(
    random_forest_model,
    X_test_sample,
    y_test_sample,
    scoring="f1_macro",
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

In [ ]:
permutation_df = pd.DataFrame({
    "Feature":
        X_test_sample.columns,

    "Importance":
        permutation_result
        .importances_mean
})

In [ ]:
top_permutation_features = (
    permutation_df
    .sort_values(
        "Importance",
        ascending=False
    )
    .head(15)
)

plt.figure(
    figsize=(10, 7)
)

sns.barplot(
    data=top_permutation_features,
    x="Importance",
    y="Feature"
)

plt.title(
    "Permutation Importance of Student Health Predictors"
)

plt.xlabel(
    "Decrease in Macro F1 After Feature Shuffling"
)

plt.ylabel(
    "Feature"
)

plt.tight_layout()
plt.show()

## Validate using cross-validation

In [ ]:
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [ ]:
cv_results = cross_validate(
    random_forest_model,
    X,
    y,
    cv=cv,

    scoring={
        "accuracy":
            "accuracy",

        "balanced_accuracy":
            "balanced_accuracy",

        "macro_f1":
            "f1_macro"
    },

    n_jobs=-1
)

In [ ]:
cv_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1"
    ],

    "Mean Score": [
        cv_results[
            "test_accuracy"
        ].mean(),

        cv_results[
            "test_balanced_accuracy"
        ].mean(),

        cv_results[
            "test_macro_f1"
        ].mean()
    ],

    "Standard Deviation": [
        cv_results[
            "test_accuracy"
        ].std(),

        cv_results[
            "test_balanced_accuracy"
        ].std(),

        cv_results[
            "test_macro_f1"
        ].std()
    ]
})

cv_summary.round(4)

## Remove weak engineered features

In [ ]:
features_to_remove = [
    "step_category",
    "exercise_category",
    "hydration_category"
]

X_reduced = df_featured.drop(
    columns=[
        "health_condition",
        *features_to_remove
    ]
)

y = df_featured[
    "health_condition"
]

In [ ]:
X_train_reduced, X_test_reduced, \
y_train_reduced, y_test_reduced = train_test_split(
    X_reduced,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
numerical_features_reduced = (
    X_train_reduced
    .select_dtypes(
        include="number"
    )
    .columns
    .tolist()
)

categorical_features_reduced = (
    X_train_reduced
    .select_dtypes(
        exclude="number"
    )
    .columns
    .tolist()
)

In [ ]:
reduced_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features_reduced
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_features_reduced
        )
    ]
)

In [ ]:
reduced_rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            reduced_preprocessor
        ),

        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=20,
                min_samples_split=10,
                min_samples_leaf=5,
                class_weight="balanced",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

In [ ]:
reduced_rf_model.fit(
    X_train_reduced,
    y_train_reduced
)

y_pred_reduced = (
    reduced_rf_model.predict(
        X_test_reduced
    )
)

In [ ]:
final_comparison = pd.DataFrame({
    "Model": [
        "Original Random Forest",
        "Reduced Random Forest"
    ],

    "Accuracy": [
        accuracy_score(
            y_test,
            y_pred_rf
        ),

        accuracy_score(
            y_test_reduced,
            y_pred_reduced
        )
    ],

    "Balanced Accuracy": [
        balanced_accuracy_score(
            y_test,
            y_pred_rf
        ),

        balanced_accuracy_score(
            y_test_reduced,
            y_pred_reduced
        )
    ],

    "Macro F1": [
        f1_score(
            y_test,
            y_pred_rf,
            average="macro"
        ),

        f1_score(
            y_test_reduced,
            y_pred_reduced,
            average="macro"
        )
    ]
})

final_comparison.round(4)

In [ ]:
import joblib

joblib.dump(
    random_forest_model,
    "student_health_risk_model.pkl"
)

In [ ]:
# Load the model
loaded_model = joblib.load(
    "student_health_risk_model.pkl"
)

In [ ]:
sample_student = X_test.iloc[[0]]

prediction = loaded_model.predict(
    sample_student
)

probabilities = loaded_model.predict_proba(
    sample_student
)

print(
    "Predicted health condition:",
    prediction[0]
)

print(
    "Class probabilities:"
)

for health_class, probability in zip(
    loaded_model
    .named_steps["classifier"]
    .classes_,

    probabilities[0]
):
    print(
        health_class,
        round(
            probability * 100,
            2
        ),
        "%"
    )

## Create a reusable prediction function

In [ ]:
def predict_student_health(
    model,
    student_data
):

    # Convert input dictionary into DataFrame
    student_df = pd.DataFrame(
        [student_data]
    )

    # Apply the same feature engineering
    student_df = create_features(
        student_df
    )

    # Predict health condition
    prediction = model.predict(
        student_df
    )[0]

    # Predict probabilities
    probabilities = model.predict_proba(
        student_df
    )[0]

    # Get class names
    classes = (
        model
        .named_steps["classifier"]
        .classes_
    )

    probability_results = {
        health_class:
        round(
            probability * 100,
            2
        )

        for health_class, probability
        in zip(
            classes,
            probabilities
        )
    }

    return (
        prediction,
        probability_results
    )

In [ ]:
student = {

    "sleep_duration": 5.5,

    "heart_rate": 82,

    "bmi": 27.5,

    "calorie_expenditure": 1900,

    "step_count": 4500,

    "exercise_duration": 20,

    "water_intake": 1.8,

    "diet_type": "non-veg",

    "stress_level": "high",

    "sleep_quality": "poor",

    "physical_activity_level":
        "sedentary",

    "smoking_alcohol": "yes",

    "gender": "male"
}

In [ ]:
prediction, probabilities = (
    predict_student_health(
        random_forest_model,
        student
    )
)

print(
    "Predicted Health Condition:",
    prediction
)

print(
    "\nPrediction Probabilities:"
)

for health_class, probability \
in probabilities.items():

    print(
        f"{health_class}: "
        f"{probability}%"
    )

In [ ]:
def get_prediction_confidence(
    probabilities
):

    confidence = max(
        probabilities.values()
    )

    if confidence >= 80:

        confidence_level = "High"

    elif confidence >= 60:

        confidence_level = "Moderate"

    else:

        confidence_level = "Low"

    return (
        confidence,
        confidence_level
    )

In [ ]:
confidence, confidence_level = (
    get_prediction_confidence(
        probabilities
    )
)

print(
    "Prediction Confidence:",
    f"{confidence}%"
)

print(
    "Confidence Level:",
    confidence_level
)

In [ ]:
def generate_recommendations(
    student
):

    recommendations = []

    if (
        student["sleep_duration"]
        < 7
    ):

        recommendations.append(
            "Increase sleep duration "
            "toward 7–9 hours."
        )

    if (
        student["stress_level"]
        == "high"
    ):

        recommendations.append(
            "Consider stress-management "
            "activities and adequate rest."
        )

    if (
        student[
            "physical_activity_level"
        ]
        == "sedentary"
    ):

        recommendations.append(
            "Increase regular physical "
            "activity gradually."
        )

    if (
        student["step_count"]
        < 5000
    ):

        recommendations.append(
            "Increase daily movement "
            "and step count gradually."
        )

    if (
        student[
            "exercise_duration"
        ]
        < 30
    ):

        recommendations.append(
            "Aim for more consistent "
            "exercise when appropriate."
        )

    if (
        student["water_intake"]
        < 2
    ):

        recommendations.append(
            "Consider increasing daily "
            "water intake."
        )

    if (
        student["sleep_quality"]
        == "poor"
    ):

        recommendations.append(
            "Improve sleep habits and "
            "maintain a consistent schedule."
        )

    if (
        student["smoking_alcohol"]
        == "yes"
    ):

        recommendations.append(
            "Reducing smoking or alcohol "
            "exposure may support health."
        )

    return recommendations

In [ ]:
recommendations = (
    generate_recommendations(
        student
    )
)

print(
    "General Wellness Suggestions:\n"
)

for number, recommendation \
in enumerate(
    recommendations,
    start=1
):

    print(
        f"{number}. "
        f"{recommendation}"
    )

In [ ]:
import joblib

joblib.dump(
    random_forest_model,
    "student_health_risk_model.pkl"
)

print(
    "Model saved successfully."
)

In [ ]:
model_metadata = {

    "model_name":
        "Student Health Risk Predictor",

    "algorithm":
        "Random Forest Classifier",

    "target":
        "health_condition",

    "classes": [
        "fit",
        "at-risk",
        "unhealthy"
    ],

    "test_accuracy":
        0.9177,

    "balanced_accuracy":
        0.9001,

    "test_macro_f1":
        0.8176,

    "cross_validation_macro_f1":
        0.8212,

    "important_predictors": [
        "stress_level",
        "physical_activity_level",
        "sleep_duration"
    ]
}

In [ ]:
import json

with open(
    "model_metadata.json",
    "w"
) as file:

    json.dump(
        model_metadata,
        file,
        indent=4
    )

## Training multiple models and comparing their accuracy

In [ ]:
from sklearn.base import clone

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import SGDClassifier, LogisticRegression


from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

from catboost import CatBoostClassifier

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import time

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),

    "Random Forest": RandomForestClassifier(),

    "Extra Trees": ExtraTreesClassifier(),

    "Decision Tree": DecisionTreeClassifier(),

    "SGD Classifier": SGDClassifier()    

}


In [ ]:
results = []

trained_models = {}

predictions = {}


for model_name, classifier in models.items():

    print(
        f"Training {model_name}..."
    )

    start_time = time.time()

    model_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor)
            ),

            (
                "classifier",
                classifier
            )
        ]
    )

    model_pipeline.fit(
        X_train,
        y_train
    )

    y_pred = model_pipeline.predict(
        X_test
    )

    training_time = (
        time.time()
        - start_time
    )

    results.append({

        "Model":
            model_name,

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_test,
                y_pred
            ),

        "Macro Precision":
            precision_score(
                y_test,
                y_pred,
                average="macro"
            ),

        "Macro Recall":
            recall_score(
                y_test,
                y_pred,
                average="macro"
            ),

        "Macro F1":
            f1_score(
                y_test,
                y_pred,
                average="macro"
            ),

        "Training Time (seconds)":
            training_time
    })

    trained_models[
        model_name
    ] = model_pipeline

    predictions[
        model_name
    ] = y_pred

    print(
        f"{model_name} completed.\n"
    )

In [ ]:
model_results = pd.DataFrame(
    results
)

model_results = (
    model_results
    .sort_values(
        by="Macro F1",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

model_results.round(4)

In [ ]:
best_model_name = (
    model_results
    .iloc[0]["Model"]
)

best_model = trained_models[
    best_model_name
]

print(
    "Best model:",
    best_model_name
)

print(
    "Best Macro F1:",
    round(
        model_results
        .iloc[0]["Macro F1"],
        4
    )
)

In [ ]:
from sklearn.preprocessing import (
    LabelEncoder
)

label_encoder = LabelEncoder()

y_train_encoded = (
    label_encoder.fit_transform(
        y_train
    )
)

y_test_encoded = (
    label_encoder.transform(
        y_test
    )
)

print(
    label_encoder.classes_
)

In [ ]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print(
    X_train_processed.shape
)

In [ ]:
boosting_models = {

    "HistGradientBoosting":
        HistGradientBoostingClassifier(
            learning_rate=0.1,
            max_iter=200,
            max_depth=10,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators=300,
            learning_rate=0.1,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            eval_metric="mlogloss",
            n_jobs=-1,
            random_state=42
        ),

    "LightGBM":
        LGBMClassifier(
            n_estimators=300,
            learning_rate=0.1,
            max_depth=-1,
            num_leaves=31,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42,
            verbosity=-1
        ),

    "CatBoost":
        CatBoostClassifier(
            iterations=300,
            learning_rate=0.1,
            depth=8,
            auto_class_weights="Balanced",
            verbose=0,
            random_seed=42
        ),

    "AdaBoost":
        AdaBoostClassifier(
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        ),
    # Very slow
    # "Gradient Boosting":GradientBoostingClassifier()
}

In [ ]:
boosting_results = []

trained_boosting_models = {}


for model_name, model in (
    boosting_models.items()
):

    print(
        f"Training {model_name}..."
    )

    start_time = time.time()

    model.fit(
        X_train_processed,
        y_train_encoded
    )

    y_pred = model.predict(
        X_test_processed
    )

    y_pred = (
        np.asarray(y_pred)
        .reshape(-1)
        .astype(int)
    )

    training_time = (
        time.time()
        - start_time
    )

    boosting_results.append({

        "Model":
            model_name,

        "Accuracy":
            accuracy_score(
                y_test_encoded,
                y_pred
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_test_encoded,
                y_pred
            ),

        "Macro Precision":
            precision_score(
                y_test_encoded,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Macro Recall":
            recall_score(
                y_test_encoded,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Macro F1":
            f1_score(
                y_test_encoded,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Training Time":
            training_time
    })

    trained_boosting_models[
        model_name
    ] = model

    print(
        f"{model_name} completed "
        f"in {training_time:.2f} seconds.\n"
    )

In [ ]:
boosting_comparison = (
    pd.DataFrame(
        boosting_results
    )
    .sort_values(
        "Macro F1",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

boosting_comparison.round(4)

In [ ]:
best_model_name = (
    boosting_comparison
    .iloc[0]["Model"]
)

best_boosting_model = (
    trained_boosting_models[
        best_model_name
    ]
)

print(
    "Best boosting model:",
    best_model_name
)

print(
    "Best Macro F1:",
    round(
        boosting_comparison
        .iloc[0]["Macro F1"],
        4
    )
)

In [ ]:
rf_prediction_encoded = (
    label_encoder.transform(
        y_pred_rf
    )
)

In [ ]:
xgb_model = (
    trained_boosting_models[
        "XGBoost"
    ]
)

xgb_prediction = (
    xgb_model.predict(
        X_test_processed
    )
)

xgb_prediction = (
    np.asarray(
        xgb_prediction
    )
    .reshape(-1)
    .astype(int)
)

In [ ]:
final_model_comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost"
    ],

    "Accuracy": [
        accuracy_score(
            y_test_encoded,
            rf_prediction_encoded
        ),

        accuracy_score(
            y_test_encoded,
            xgb_prediction
        )
    ],

    "Balanced Accuracy": [
        balanced_accuracy_score(
            y_test_encoded,
            rf_prediction_encoded
        ),

        balanced_accuracy_score(
            y_test_encoded,
            xgb_prediction
        )
    ],

    "Macro Precision": [
        precision_score(
            y_test_encoded,
            rf_prediction_encoded,
            average="macro"
        ),

        precision_score(
            y_test_encoded,
            xgb_prediction,
            average="macro"
        )
    ],

    "Macro Recall": [
        recall_score(
            y_test_encoded,
            rf_prediction_encoded,
            average="macro"
        ),

        recall_score(
            y_test_encoded,
            xgb_prediction,
            average="macro"
        )
    ],

    "Macro F1": [
        f1_score(
            y_test_encoded,
            rf_prediction_encoded,
            average="macro"
        ),

        f1_score(
            y_test_encoded,
            xgb_prediction,
            average="macro"
        )
    ]
})

final_model_comparison.round(4)

In [ ]:
print(
    classification_report(
        y_test_encoded,
        xgb_prediction,
        target_names=
            label_encoder.classes_
    )
)

In [ ]:
xgb_cm = confusion_matrix(
    y_test_encoded,
    xgb_prediction
)

plt.figure(
    figsize=(8, 6)
)

sns.heatmap(
    xgb_cm,
    annot=True,
    fmt=",",
    cmap="Blues",
    xticklabels=
        label_encoder.classes_,
    yticklabels=
        label_encoder.classes_
)

plt.title(
    "XGBoost Confusion Matrix"
)

plt.xlabel(
    "Predicted Health Condition"
)

plt.ylabel(
    "Actual Health Condition"
)

plt.tight_layout()
plt.show()

## Hyperparameter tuning